In [6]:
import numpy as np
import pandas as pd
from utils import load_data

df = load_data()
df["series_id"] = df["SKU_ID"] + "_" + df["Warehouse_ID"]

forecast = pd.read_csv("forecast_mysql.csv", parse_dates=["forecast_date"])
forecast["series_id"] = forecast["sku_id"] + "_" + forecast["warehouse_id"]
print("forecast:", forecast.shape)

# the day our forecast starts from (last training day)
as_of = pd.Timestamp("2024-11-30")

# stock and supplier facts of each series on that day
facts = df[df["Date"] == as_of][["series_id", "Inventory_Level", "Supplier_Lead_Time_Days",
                                  "Reorder_Point", "Unit_Cost", "Unit_Price"]]
facts = facts[facts["series_id"].isin(forecast["series_id"].unique())].reset_index(drop=True)
facts

forecast: (900, 9)


,series_id,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Unit_Cost,Unit_Price
0,SKU_13_WH_2,399,11,350,17.04,21.69
1,SKU_18_WH_4,354,6,250,5.35,9.51
2,SKU_18_WH_5,853,13,394,13.35,21.48
3,SKU_20_WH_1,497,9,295,9.55,16.05
4,SKU_22_WH_3,261,4,211,18.38,32.58
5,SKU_23_WH_5,528,3,383,19.43,28.74
6,SKU_29_WH_1,512,5,218,7.00,11.27
7,SKU_29_WH_2,740,14,346,15.38,19.90
8,SKU_33_WH_4,438,3,274,9.27,14.74
9,SKU_47_WH_3,608,10,377,7.79,13.38


In [7]:
Z = 1.65   # 95% service level

rows = []
for _, f in facts.iterrows():
    sid = f["series_id"]
    past = df[(df["series_id"] == sid) & (df["Date"] <= as_of)]["Units_Sold"]
    fc = forecast[(forecast["series_id"] == sid) & (forecast["model_name"] == "ARIMA")]

    lead = f["Supplier_Lead_Time_Days"]
    avg_demand = fc["forecast_units"].mean()
    std_demand = past.tail(90).std()
    safety_stock = Z * std_demand * np.sqrt(lead)

    rows.append({
        "series_id": sid,
        "current_stock": f["Inventory_Level"],
        "lead_time_days": lead,
        "Reorder_Point": f["Reorder_Point"],
        "Unit_Cost": f["Unit_Cost"],
        "Unit_Price": f["Unit_Price"],
        "avg_daily_demand": avg_demand,
        "demand_std": std_demand,
        "safety_stock": safety_stock,
        "reorder_point_new": avg_demand * lead + safety_stock,
    })

inv = pd.DataFrame(rows).round(1)
inv

,series_id,current_stock,lead_time_days,Reorder_Point,Unit_Cost,Unit_Price,avg_daily_demand,demand_std,safety_stock,reorder_point_new
0,SKU_13_WH_2,399,11,350,17.0,21.7,13.5,5.4,29.7,178.6
1,SKU_18_WH_4,354,6,250,5.4,9.5,15.1,5.2,20.9,111.3
2,SKU_18_WH_5,853,13,394,13.4,21.5,13.4,4.6,27.3,200.9
3,SKU_20_WH_1,497,9,295,9.6,16.0,17.1,5.3,26.3,180.1
4,SKU_22_WH_3,261,4,211,18.4,32.6,14.1,5.9,19.6,75.9
5,SKU_23_WH_5,528,3,383,19.4,28.7,16.0,5.3,15.2,63.1
6,SKU_29_WH_1,512,5,218,7.0,11.3,14.7,4.8,17.7,91.1
7,SKU_29_WH_2,740,14,346,15.4,19.9,14.7,5.3,33.0,238.8
8,SKU_33_WH_4,438,3,274,9.3,14.7,14.1,5.2,14.9,57.2
9,SKU_47_WH_3,608,10,377,7.8,13.4,12.7,5.0,26.0,152.9


In [8]:
inv["days_of_stock"] = inv["current_stock"] / inv["avg_daily_demand"]

def risk(row):
    if row["days_of_stock"] < row["lead_time_days"]:
        return "RED"
    elif row["current_stock"] < row["reorder_point_new"]:
        return "YELLOW"
    return "GREEN"

inv["risk"] = inv.apply(risk, axis=1)

inv["excess_units"] = inv["current_stock"] - inv["reorder_point_new"]
inv["excess_value"] = inv["excess_units"] * inv["Unit_Cost"]
inv["days_until_reorder"] = inv["excess_units"] / inv["avg_daily_demand"]
inv = inv.round(1)

print(inv["risk"].value_counts())
display(inv[["series_id", "current_stock", "reorder_point_new", "days_of_stock",
             "days_until_reorder", "excess_units", "excess_value", "risk"]]
        .sort_values("days_until_reorder"))
print("Total money tied up in extra stock:", round(inv["excess_value"].sum()))

# save full table
out = inv.rename(columns={"Reorder_Point": "reorder_point_dataset",
                          "reorder_point_new": "reorder_point"})
out.to_csv("inventory_risk.csv", index=False)

# file for SQL
def split_series(d):
    d = d.copy()
    parts = d["series_id"].str.split("_WH_", expand=True)
    d.insert(0, "sku_id", parts[0])
    d.insert(1, "warehouse_id", "WH_" + parts[1])
    return d.drop(columns="series_id")

cols = ["series_id", "current_stock", "avg_daily_demand", "demand_std", "safety_stock",
        "reorder_point", "days_of_stock", "days_until_reorder",
        "excess_units", "excess_value", "risk"]
ia = split_series(out[cols])
ia.to_csv("inventory_sql.csv", index=False)
print("saved inventory_sql.csv:", ia.shape)   # expect (10, 12)

risk
GREEN    10
Name: count, dtype: int64


,series_id,current_stock,reorder_point_new,days_of_stock,days_until_reorder,excess_units,excess_value,risk
4,SKU_22_WH_3,261,75.9,18.5,13.1,185.1,3405.8,GREEN
1,SKU_18_WH_4,354,111.3,23.4,16.1,242.7,1310.6,GREEN
0,SKU_13_WH_2,399,178.6,29.6,16.3,220.4,3746.8,GREEN
3,SKU_20_WH_1,497,180.1,29.1,18.5,316.9,3042.2,GREEN
8,SKU_33_WH_4,438,57.2,31.1,27.0,380.8,3541.4,GREEN
6,SKU_29_WH_1,512,91.1,34.8,28.6,420.9,2946.3,GREEN
5,SKU_23_WH_5,528,63.1,33.0,29.1,464.9,9019.1,GREEN
7,SKU_29_WH_2,740,238.8,50.3,34.1,501.2,7718.5,GREEN
9,SKU_47_WH_3,608,152.9,47.9,35.8,455.1,3549.8,GREEN
2,SKU_18_WH_5,853,200.9,63.7,48.7,652.1,8738.1,GREEN


Total money tied up in extra stock: 47019
saved inventory_sql.csv: (10, 12)
